# Inspect the results of the assessment of a classifier's fairness based on movement patterns

In [ ]:
import pandas as pd
import numpy as np
import pickle

import folium
import branca.colormap as cm

In [ ]:
# Read the flattened candidates.
path_dict_candidates = './data_simulator/huge_dataset/gencand/dict_flattened_candidates.pkl'
with open(path_dict_candidates, "rb") as f:
    dict_candidates = pickle.load(f)

# Read the results of an assessment of fairness based on movement patterns from disk.
path_results = './res_exp.pkl'
with open(path_results, "rb") as f:
    dict_res = pickle.load(f)

# Read the geodataframes of the grids (needed to plot the results on a map).
path_dict_grids = './data_simulator/huge_dataset/grids/dict_grids.pkl'
with open(path_dict_grids, "rb") as f:
    dict_grids = pickle.load(f)

For each candidate, retrieve the grid and subset of cell it refers to.

In [ ]:
# Retrieve the set of candidates (subset of cells) that underwent hypothesis testing.
grid_info = dict_candidates['grid_info']
# display(grid_info)

# Compute the number of objects associated with each candidate
num_objs_candidates = np.diff(dict_candidates['start_pos'])

# Retrieve the log-likelihood ratios computed for the candidates.
extreme_lr_threshold = dict_res['threshold_extreme']
vec_lr_dataset = dict_res['vec_LR_dataset']
vec_inrate_dataset = dict_res['vec_inrate_dataset']
vec_outrate_dataset = dict_res['vec_outrate_dataset']
labels = dict_res['dataset']
# display(vec_lr_dataset)


# For each candidate, here represented as a tuple of cell IDs, associate the grid and subset of cells it refers to.
num_candidates = vec_lr_dataset.size
list_grid_ids = np.empty(num_candidates, dtype=object)
list_cellids = np.empty(num_candidates, dtype=object)
count = 0
for grid in grid_info:
    cell_ids = grid[3].to_numpy()
    num_els_grid = cell_ids.size

    grid_id = np.empty(1, dtype=object)
    grid_id[0] = (int(grid[1]), int(grid[2]))
    grid_id = np.repeat(grid_id, cell_ids.size)    
    
    list_grid_ids[count : count + num_els_grid] = grid_id
    list_cellids[count : count + num_els_grid] = cell_ids

    count += num_els_grid


# Put all the information in a pandas Dataframe.
df_candidates = pd.DataFrame({
    "grid_id":   list_grid_ids,
    "cell_ids":  list_cellids,
    "lr":        vec_lr_dataset.astype(np.float32),
    "num_objs":  num_objs_candidates.astype(np.uint32),
    "in_rate":   vec_inrate_dataset.astype(np.float32),
    "out_rate":  vec_outrate_dataset.astype(np.float32)
})
#
# Count the number of cells making up each candidate.
# NOTE: We use numpy's 'fromiter' because it's way faster (C-backed code) than using pandas' apply/map + lamba func on a series.
cell_ids = df_candidates['cell_ids'].to_numpy()
df_candidates['num_cells'] = np.fromiter(
    (len(x) if type(x) is tuple else 1 for x in cell_ids),
    dtype=np.uint32,
    count=cell_ids.size
)
del cell_ids
print(df_candidates)


# Free some memory.
del list_grid_ids, list_cellids, vec_lr_dataset, num_objs_candidates, vec_inrate_dataset, vec_outrate_dataset
del dict_candidates, dict_res

In [ ]:
# Pick a grid and select the candidates (subsets of cells) that are made of a certain number of cells.
resolution, offset = 1000, 0
target_grid = (resolution, offset)
target_numcells_candidate = 1

# Select the candidates belonging to the 'target' grid.
df_sel_candidates = df_candidates.loc[df_candidates['grid_id'] == target_grid].copy()
# display(df_sel_candidates)

# Select the candidates with the desired number of cells.
df_sel_candidates = df_sel_candidates.loc[df_sel_candidates['num_cells'] == target_numcells_candidate]
#display(df_sel_candidates.sort_values(['in_rate', 'num_objs']).tail(30))
#display(df_sel_candidates.sort_values(['lr']).tail(30))
display(df_sel_candidates)

In [ ]:
# Retrieve the geopandas dataframe of the grid of interest.
grid = dict_grids[target_grid].grid.copy()
# display(grid)


# Augment the dataframe with various info.
grid.loc[:, ['lr', 'in', 'out']], grid.loc[:, 'nobjs'] = np.float32(0.), np.uint32(0)
sel_cell_ids = df_sel_candidates['cell_ids']
grid.loc[df_sel_candidates['cell_ids'], 'lr'] = df_sel_candidates['lr'].values
grid.loc[df_sel_candidates['cell_ids'], 'nobjs'] = df_sel_candidates['num_objs'].values
grid.loc[df_sel_candidates['cell_ids'], 'in'] = df_sel_candidates['in_rate'].values
grid.loc[df_sel_candidates['cell_ids'], 'out'] = df_sel_candidates['out_rate'].values

# Determine which cells have an extreme log-likelihood ratio, according to the simulations' results.
grid['is_extreme'] = grid['lr'] >= extreme_lr_threshold

grid

### DEBUG: inspect the results of 1-cell candidates with plots...

Plot the considered grid on a map, showing a heatmap of the inside positive rates of each grid's cell.

**NOTE**: The cells below work only when considering candidates made of just 1 cell.

In [ ]:
# Center map on grid bounds
minx, miny, maxx, maxy = grid.total_bounds
m = folium.Map(
    location=[(miny + maxy) / 2, (minx + maxx) / 2],
    zoom_start=12,
    control_scale=True
)

# Build a continuous colormap from lr values
grid["cell_id"] = grid.index.astype(str)
vmin = float(grid["in"].min())
vmax = float(grid["in"].max())
colormap = cm.linear.YlOrRd_09.scale(vmin, vmax)  # pick any palette you like
colormap.caption = "positive rate"
colormap.add_to(m)

def style_fn(feature):
    val = feature["properties"].get("in", None)
    return {
        "color": "blue",
        "weight": 1,
        "fillColor": colormap(float(val)),
        "fillOpacity": 0.7,
    }

folium.GeoJson(
    grid,
    style_function=style_fn,
    tooltip=folium.GeoJsonTooltip(fields=["cell_id", "lr", "nobjs", "in", "out"], 
                                  aliases=["cell ID:", "lr:", "num_objs:", "pr_in:", "pr_out:"]),
).add_to(m)

m

Plot the considered grid on a map, showing a heatmap of the log-likelihood ratios of each grid's cell.

**NOTE**: The cells below work only when considering candidates made of just 1 cell.

In [ ]:
# Center map on grid bounds
minx, miny, maxx, maxy = grid.total_bounds
m = folium.Map(
    location=[(miny + maxy) / 2, (minx + maxx) / 2],
    zoom_start=12,
    control_scale=True
)

# Build a continuous colormap from lr values
grid["cell_id"] = grid.index.astype(str)
vmin = float(grid["lr"].min())
vmax = float(grid["lr"].max())
colormap = cm.linear.YlOrRd_09.scale(vmin, vmax)  # pick any palette you like
colormap.caption = "log-likelihood ratio"
colormap.add_to(m)

def style_fn(feature):
    val = feature["properties"].get("lr", None)
    return {
        "color": "blue",
        "weight": 1,
        "fillColor": colormap(float(val)),
        "fillOpacity": 0.7,
    }

folium.GeoJson(
    grid,
    style_function=style_fn,
    tooltip=folium.GeoJsonTooltip(fields=["cell_id", "lr", "nobjs", "in", "out"], 
                                  aliases=["cell ID:", "lr:", "num_objs:", "pr_in:", "pr_out:"]),
).add_to(m)

m

Plot the considered grid on a map, showing where the cells with extreme log-likelihood ratios are located.

**NOTE**: The cells below work only when considering candidates made of just 1 cell.

In [ ]:
# Center map on grid bounds
minx, miny, maxx, maxy = grid.total_bounds
m = folium.Map(
    location=[(miny + maxy) / 2, (minx + maxx) / 2],
    zoom_start=12,
    control_scale=True
)

# Base grid (light)
folium.GeoJson(
    grid,
    style_function=lambda f: {
        "color": "#666666",
        "weight": 0.6,
        "fillColor": "#d9d9d9",
        "fillOpacity": 0.15,
    },
    tooltip=folium.GeoJsonTooltip(fields=["cell_id"]),
).add_to(m)

# Extreme cells (highlighted)
extreme_cells = grid[grid["is_extreme"] == True]
folium.GeoJson(
    extreme_cells,
    style_function=lambda f: {
        "color": "red",
        "weight": 2,
        "fillColor": "black",
        "fillOpacity": 0.65,
    },
    tooltip=folium.GeoJsonTooltip(fields=["cell_id", "lr", "nobjs", "in", "out"]),
    name="Extreme cells",
).add_to(m)

folium.LayerControl().add_to(m)
m

### DEBUG: results' sanity check!

Check that (1) the number of objects, (2) inside rate, and (3) outside rate that can be derived from the candidates' original files, align with those computed during the hypothesis test (which uses the candidates' flattened data structures). If these quantities align, then the the results from the hypothesis test that are derived from them (i.e., log likelihoods) are OK. 

**NOTE**: The code below is deactivated by default.